- 단순 유사도(ROUGE/BERTScore)로 잡히지 않는 **사실 검증 오류**를 주요 지표로 반영.
- 문서 가독성과 표현 품질까지 보완적으로 평가 가능.
- 실제 R&D 계획서, 규제 문서 평가 시 **신뢰성 + 완성도**를 함께 담보.

**문서 점수**

- 정합성 70%
- 표현 30%

**임계:**

- **합격**: Sfinal≥85S_{\text{final}}\ge 85Sfinal≥85 & Ccontra≤3%C_{\text{contra}}\le 3\%Ccontra≤3%
- **재검토**: 70≤Sfinal<8570\le S_{\text{final}}<8570≤Sfinal<85 또는 Cmiss>10%C_{\text{miss}}>10\%Cmiss>10%
- **반려**: Sfinal<70S_{\text{final}}<70Sfinal<70 또는 Ccontra>10%C_{\text{contra}}>10\%Ccontra>10%

In [19]:
import os
import json
import docx
from collections import namedtuple

# DocParser: .docx 파일의 실제 내용을 파싱
class DocParser:
    def parse(self, docx_path):
        try:
            doc = docx.Document(docx_path)
            full_text = [para.text for para in doc.paragraphs]
            sentences = []
            for para in full_text:
                for sentence in para.split('.'):
                    if sentence.strip():
                        sentences.append(sentence.strip() + '.')
            
            text_content = "\n".join(full_text)
            sections = []
            paragraphs = full_text

            Doc = namedtuple('Doc', ['sections', 'paragraphs', 'sentences', 'text'])
            return Doc(sections=sections, paragraphs=paragraphs, sentences=sentences, text=text_content)

        except FileNotFoundError:
            return None

# HybridRetriever: 근거 문헌 JSON 파일을 실제로 로드
class HybridRetriever:
    def __init__(self, cfg):
        self.rag_chunks_path = cfg["inputs"].get("rag_chunks")
        self.guidelines_path = cfg["inputs"].get("guidelines_json")
        
        self.rag_data = self._load_json(self.rag_chunks_path)
        self.guidelines_data = self._load_json(self.guidelines_path)

    def _load_json(self, file_path):
        if not file_path or not os.path.exists(file_path):
            return []
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception:
            return []

    def topk(self, claim):
        retrieved_evidences = []
        for item in self.rag_data:
            if isinstance(item, str) and claim in item:
                retrieved_evidences.append(item)
            elif isinstance(item, dict) and claim in item.get('text', ''):
                 retrieved_evidences.append(item.get('text'))
        
        if not retrieved_evidences:
            retrieved_evidences.append(f"가이드라인 또는 RAG 데이터에서 '{claim}' 관련 내용 없음.")
        
        return retrieved_evidences

# 주요 오류 평가 관련 클래스 및 함수 (더미 구현)
def extract_claims(doc):
    return [{'sent': '데이터 오류율을 20% 감소시키고, 데이터 완전성을 95% 이상으로 유지합니다.', 'type': 'KPI', 'value': '20%'}]

def select_best_evidence(claim, evidences):
    return evidences[0]

def normalize_verdict(nli, qa=None):
    return {"verdict": "entailment", "confidence": nli['max_p']}

def compute_major_metrics(results, cfg):
    return {"accuracy": 0.9}

# 사소 오류 평가 관련 클래스 및 함수 (더미 구현)
def coherence_score(sentences):
    return 0.92

def redundancy_score(sentences):
    return 0.85

def format_score(sections, required):
    return 0.98

# 기타 유틸리티 함수
def aggregate_scores(major, minor, cfg):
    final_score = (major["accuracy"] * 0.6) + ((minor["fluency"] + minor["coherence"] + minor["bart_coherence"]) / 3 * 0.4)
    return {"final_score": final_score}

def build_report(doc, results, major, minor, final):
    report_content = "## 문서 평가 보고서\n\n"
    report_content += f"### 최종 점수: {final['final_score']:.2f}\n\n"
    
    report_content += "### 주요 오류 분석 (Major Errors)\n"
    report_content += f"- 사실 관계 정확도 (Accuracy): {major['accuracy']:.2f}\n"
    report_content += "#### 개별 주장 검증 결과\n"
    for r in results:
        report_content += f"- 주장: {r['sent']}\n"
        report_content += f"  - 근거: {r['evidences']}\n"
        report_content += f"  - 평가: {r['verdict']} (신뢰도: {r['confidence']:.2f})\n\n"

    report_content += "### 사소 오류 분석 (Minor Errors)\n"
    for metric, score in minor.items():
        if isinstance(score, (int, float)):
            report_content += f"- {metric.capitalize()}: {score:.2f}\n"

    return report_content

def save_report(report, output_dir, file_name):
    report_path = os.path.join(output_dir, file_name)
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report)
    print(f"Report saved to {report_path}")


# 평가 모듈 클래스들 (더미 구현)
class NLIModel:
    def __init__(self, cfg):
        pass
    def predict(self, claim, best):
        return {'entail': 0.7, 'contra': 0.1, 'neutral': 0.2, 'max_p': 0.7}

class BooleanQA:
    def __init__(self, cfg):
        pass
    def yesno(self, question, context):
        return "yes"

class LinguisticLinter:
    def fluency_score(self, sentences):
        return 0.95

class TerminologyChecker:
    def __init__(self, glossary):
        self.glossary = glossary
        if not self.glossary:
            print("Warning: Domain glossary file not found. Terminology consistency check will be skipped.")

    def consistency_score(self, text):
        if not self.glossary:
            return 0.0
        return 0.90

class BARTScoreEvaluator:
    def __init__(self):
        pass
    def score_coherence(self, sentences):
        return 0.88
    def score_relevance(self, claims, evidences):
        return 0.92

# DocEvaluator 클래스
class DocEvaluator:
    def __init__(self, cfg):
        self.cfg = cfg
        self.parser = DocParser()
        self.retriever = HybridRetriever(cfg)
        self.nli = NLIModel(cfg["models"]["nli"])
        self.boolqa = BooleanQA(cfg["models"]["qna"])
        self.lint = LinguisticLinter()
        self.terminology = TerminologyChecker(self._load_glossary(cfg["inputs"].get("domain_glossary")))
        self.bart_score = BARTScoreEvaluator()
    
    def _load_glossary(self, file_path):
        if not file_path or not os.path.exists(file_path):
            return None
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            return None

    def evaluate(self, docx_path) -> dict:
        doc = self.parser.parse(docx_path)
        if not doc:
            return {"final_score": 0.0, "error": "Document parsing failed"}

        claims = extract_claims(doc)
        results = []
        for c in claims:
            evidences = self.retriever.topk(c["sent"])
            verdict = self.verify_claim(c["sent"], evidences)
            results.append({**c, **verdict, "evidences": evidences})

        major_scores = compute_major_metrics(results, self.cfg)
        minor_scores = self.evaluate_minor(doc)
        final = aggregate_scores(major_scores, minor_scores, self.cfg)

        file_name = os.path.basename(docx_path).replace('.docx', '_report.txt')
        report = build_report(doc, results, major_scores, minor_scores, final)
        save_report(report, self.cfg["output_dir"], file_name)
        return final

    def verify_claim(self, claim, evidences):
        best = select_best_evidence(claim, evidences)
        nli = self.nli.predict(claim, best)
        if nli['max_p'] < 0.6:
            qa = self.boolqa.yesno(question=f"Is the claim supported? {claim}", context=best)
        return normalize_verdict(nli, qa=None)

    def evaluate_minor(self, doc):
        fluency = self.lint.fluency_score(doc.sentences)
        coherence = coherence_score(doc.sentences)
        terminology = self.terminology.consistency_score(doc.text)
        redundancy = redundancy_score(doc.sentences)
        fmt = format_score(doc.sections, required=['연구개발 목표', '연구개발 내용', ...])
        
        bart_coherence_score = self.bart_score.score_coherence(doc.sentences)
        
        return {"fluency": fluency, "coherence": coherence,
                "terminology": terminology, "redundancy": redundancy, "format": fmt,
                "bart_coherence": bart_coherence_score}

# 메인 실행 블록
config = {
    "models": {
        "embed": "BAAI/bge-m3",
        "nli": "microsoft/deberta-v3-large",
        "qna": "google/flan-t5-large"
    },
    "inputs": {
        "rag_chunks": "/home/alpaco/autosry/rag_chunks.json",
        "guidelines_json": "/home/alpaco/autosry/rnd_guideline.json",
        "domain_glossary": None
    },
    "output_dir": "./eval_reports"
}

if not os.path.exists(config["output_dir"]):
    os.makedirs(config["output_dir"])

evaluator = DocEvaluator(config)

doc1_path = "/home/alpaco/autosry/post_test/cde_연구계발계획서.docx"
print(f"\n'{doc1_path}' 문서 평가를 시작합니다...")
result1 = evaluator.evaluate(doc1_path)
print(f"'{doc1_path}' 문서 평가 완료. 최종 점수: {result1['final_score']:.2f}")

doc2_path = "/home/alpaco/autosry/post_test/e5_연구계발계획서v6.docx"
print(f"\n'{doc2_path}' 문서 평가를 시작합니다...")
result2 = evaluator.evaluate(doc2_path)
print(f"'{doc2_path}' 문서 평가 완료. 최종 점수: {result2['final_score']:.2f}")


'/home/alpaco/autosry/post_test/cde_연구계발계획서.docx' 문서 평가를 시작합니다...
Report saved to ./eval_reports/cde_연구계발계획서_report.txt
'/home/alpaco/autosry/post_test/cde_연구계발계획서.docx' 문서 평가 완료. 최종 점수: 0.91

'/home/alpaco/autosry/post_test/e5_연구계발계획서v6.docx' 문서 평가를 시작합니다...
Report saved to ./eval_reports/e5_연구계발계획서v6_report.txt
'/home/alpaco/autosry/post_test/e5_연구계발계획서v6.docx' 문서 평가 완료. 최종 점수: 0.91
